# 1. Evaluating GPT

In [24]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1.1 Generating Text

In [1]:
import torch
from gpt_model import GPTModel

In [2]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False,      # Query-key-value bias
    "weight_tying": False
}

In [3]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();  # Disable dropout during inference

![](images/model-training-1.png)

### Helper Functions for Encoding and Decoding Text

In [4]:
import tiktoken
from generate_text import generate_text_simple

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    # add batch dimension
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    # remove batch dimension
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

In [5]:
start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    current_context=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you rentingetic wasnم refres RexMeCHicular stren


## 1.2 Calculating the quality of the generation - cross-entropy and perplexity

Basically, for each input set, the target output is known.
Example: For "I am 24 years", the expected output is "am 24 years old".

To calculate the loss on this sample, we
* pass the input through the model
* receive a 4 x 50257 output, where the rows represent the index position and the columns represent the logits for each vocab token
* apply a softmax to the logits at each index position to get probability scores
* check the probability scores for the correct next tokens
    * "am" in index 0
    * "24" in index 1
    * "years" in index 2
    * "old" in index 3
* the farther these individual probabilities are from 1, the higher the cross-entropy loss

![](images/model-training-3.png)

In [6]:
# Define inputs and the corresponding targets

inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107,  588, 11311]]) #  " really like chocolate"]

In [7]:
# Pass the inputs through the model and get the predictions

with torch.no_grad():
    logits = model(inputs)

# logits will be of the form: [batch_size, num_tokens, vocab_size]
# i.e., [2, 3, 50257]
# so, we compute the softmax for each token position along the last dimension
probas = torch.softmax(logits, dim=-1)

print(probas.shape)

torch.Size([2, 3, 50257])


Converting the probability scores back into text

![](images/model-training-2.png)

In [8]:
# Pick the token with the max probability as the "predicted" outputs

predicted_token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print("Token IDs:\n", predicted_token_ids)

Token IDs:
 tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]])


In [9]:
# Compare the actual vs predicted outputs

print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1: {token_ids_to_text(predicted_token_ids[0].flatten(), tokenizer)}")

print(f"Targets batch 2: {token_ids_to_text(targets[1], tokenizer)}")
print(f"Outputs batch 2: {token_ids_to_text(predicted_token_ids[1].flatten(), tokenizer)}")

Targets batch 1:  effort moves you
Outputs batch 1:  Armed heNetflix
Targets batch 2:  really like chocolate
Outputs batch 2:  pressuring empoweredfaith


In [10]:
# Inspect the probabilities for the actual target tokens

# Gimme the probabilities for token IDs targets[0][0], 
# targets[0][1] and targets[0][2] at index 0, 1 and 2
# respectively
target_probas_1 = probas[0, [0, 1, 2], targets[0]]
print("Text 1:", target_probas_1)

# Gimme the probabilities for token IDs targets[1][0], 
# targets[1][1] and targets[1][2] at index 0, 1 and 2
# respectively
target_probas_2 = probas[1, [0, 1, 2], targets[1]]
print("Text 2:", target_probas_2)

Text 1: tensor([7.4541e-05, 3.1061e-05, 1.1563e-05])
Text 2: tensor([1.0337e-05, 5.6776e-05, 4.7559e-06])


We wanna maximize the above values, which would mean the model very strongly predicted what we actually wanted it to predict!

In [11]:
# Compute the log loss across all tokens
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print(log_probas)

tensor([ -9.5042, -10.3796, -11.3677, -11.4798,  -9.7764, -12.2561])


In [12]:
# Average it out
avg_log_probas = torch.mean(log_probas)
print(avg_log_probas)

tensor(-10.7940)


We wanna maximize this value! Since it's a log, the max value is 0

In [13]:
# As per ML convention, minimize the average log probability

neg_avg_log_probas = avg_log_probas * -1
print(neg_avg_log_probas)

tensor(10.7940)


### Cross-entropy

PyTorch already implements a cross_entropy function that carries out the previous steps

Cross-entropy calculates how far two probability distributions are from each other.

This single "average loss" is what the optimizer uses to update the model’s weights via backpropagation.

**Lower is better**

![](images/model-training-3.png)

In [14]:
# Logits have shape (batch_size, num_tokens, vocab_size)
print("Logits shape:", logits.shape)

# Targets have shape (batch_size, num_tokens)
print("Targets shape:", targets.shape)

Logits shape: torch.Size([2, 3, 50257])
Targets shape: torch.Size([2, 3])


In [15]:
# For the cross_entropy function in PyTorch, we want to flatten these tensors 
# by combining them over the batch dimension

logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()

print("Flattened logits:", logits_flat.shape)
print("Flattened targets:", targets_flat.shape)

Flattened logits: torch.Size([6, 50257])
Flattened targets: torch.Size([6])


In [16]:
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)

tensor(10.7940)


* The `cross_entropy` function knows the targe token IDs we ideally wanted (the second param). 
* It then looks at the logits for each of the 6 tokens and inspects the compute probs for these token IDs. 
* It then gives us the loss value, which represents how far away the model was from predicting really high values
corresponding to the tokens that we actually wanted.

### Perplexity

Related term which measures how well the probability distribution predicted by the model matches the actual distribution of the words in the dataset.

**Example: if perplexity is 48725, then it means that at each step, that the model picked the next token as if picking randomly from 48725 options.**

**It is simply = e^cross-entropy**

**Lower is better.**

In [17]:
perplexity = torch.exp(loss)
print(perplexity)

tensor(48725.8203)


## 1.3 Calculating this loss on training and validation sets

We use a smaller dataset for this colab (just one short story), since:
* it can run quickly on my personal machine
* public domain text, so no violations of permissions
* for reference, llama 2-7B requried 184,320 GPU hours on A100 GPUS and was trained on 2 trillion tokens (although, as per the Chinchilla paper, 7*20 = 140Bn tokens might've been enough?)
    *  Hourly cost of each A100 server = $3.75
    *  Hence, total = 184,320 * 3.75 = $691,200
    *  Hence, approx $700k

In [18]:
# Load the dataset from our repo

import os
import requests

file_path = "the-verdict.txt"

with open(file_path, "r", encoding="utf-8") as file:
    text_data = file.read()

# make sure the data loaded properly
print(text_data[:99])
print(text_data[-99:])

total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))

print("Characters:", total_characters)
print("Tokens:", total_tokens)

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 
it for me! The Strouds stand alone, and happen once--but there's no exterminating our kind of art."
Characters: 20479
Tokens: 5145


![](images/model-training-4.png)

### Explain the batch sizing?

* Batch Size: Determines how many independent samples the model sees in one training step (e.g., batch_size=4).
* Dimensions: A single batch is a tensor of shape (Batch Size, Context Length).
* Total Batches: Calculated as $\frac{\text{Total Samples}}{\text{Batch Size}}$. Since we set `stride = context_length` in the following, each token in the dataset appears in exactly one input sample, minimizing the total number of batches to the most efficient count.

### How is each input, target pair related?

For any given sequence, the Target is simply the Input shifted by one position to the right.
* Input IDs: [0, 1, 2, ..., 255] (Indices 0 to 256)
* Target IDs: [1, 2, 3, ..., 256] (Indices 1 to 257)
* The Logic: At every index $i$ in the input, the model's goal is to predict the token at index $i$ of the target. For example, the token at input[0] is used to predict the token at target[0].

### How are stride and context length related?

The stride determines how much the "window" slides across the text.

* Zero Overlap (stride = context_length): * Why: Maximum efficiency. Every token transition is seen exactly once. It’s the fastest way to train on large datasets without redundancy.

* 50% Overlap (stride = context_length // 2): * Why: Provides "Contextual Variety." A token that appeared at the very end of Sample A (with 255 tokens of "past" context) will appear in the middle of Sample B, helping the model learn to predict it more robustly.

* Near 100% Overlap (stride = 1): * Why: Used only for extremely small datasets. It "recycles" the data to create the maximum possible number of samples, though it risks high redundancy and overfitting.

### Some sample indices for the input, output pairs when stride = context_length

* Creating an input chunk from index 0, 256 (1)
* Creating an output chunk from index 1, 257
* Creating an input chunk from index 256, 512 (2)
* Creating an output chunk from index 257, 513
* Creating an input chunk from index 512, 768 (3) 
* Creating an output chunk from index 513, 769
* Creating an input chunk from index 768, 1024 (4)
* Creating an output chunk from index 769, 1025

In [28]:
# Split into training and validation sets

# Since we train the LLM to predict the next word in the text, 
# the targets look the same as these inputs, 
# except that the targets are shifted by one position

from data_loader import create_dataloader_v1

# Train/validation ratio
train_ratio = 0.90

# Split the input text into 90-10
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    # We use a relatively small batch size to reduce the computational 
    # resource demand, and because the dataset is very small to begin with
    batch_size=2,
    context_size=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    # We use a relatively small batch size to reduce the computational 
    # resource demand, and because the dataset is very small to begin with
    batch_size=2,
    context_size=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

Creating an input chunk from index 0, 256 
Creating an output chunk from index 1, 257 
Creating an input chunk from index 256, 512 
Creating an output chunk from index 257, 513 
Creating an input chunk from index 512, 768 
Creating an output chunk from index 513, 769 
Creating an input chunk from index 768, 1024 
Creating an output chunk from index 769, 1025 
Creating an input chunk from index 1024, 1280 
Creating an output chunk from index 1025, 1281 
Creating an input chunk from index 1280, 1536 
Creating an output chunk from index 1281, 1537 
Creating an input chunk from index 1536, 1792 
Creating an output chunk from index 1537, 1793 
Creating an input chunk from index 1792, 2048 
Creating an output chunk from index 1793, 2049 
Creating an input chunk from index 2048, 2304 
Creating an output chunk from index 2049, 2305 
Creating an input chunk from index 2304, 2560 
Creating an output chunk from index 2305, 2561 
Creating an input chunk from index 2560, 2816 
Creating an output ch

In [20]:
# Sanity checks to make sure the data was loaded correctly

if total_tokens * (train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the training loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "increase the `training_ratio`")

if total_tokens * (1-train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the validation loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "decrease the `training_ratio`")

print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\nValidation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

Train loader:
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])

Validation loader:
torch.Size([2, 256]) torch.Size([2, 256])


In [21]:
# Some stats about the loaded data

train_tokens = 0
for input_batch, target_batch in train_loader:
    train_tokens += input_batch.numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
    val_tokens += input_batch.numel()

print("Training tokens:", train_tokens)
print("Validation tokens:", val_tokens)
print("All tokens:", train_tokens + val_tokens)

Training tokens: 4608
Validation tokens: 512
All tokens: 5120


Implementing a utility function to calculate the cross-entropy loss of a given batch

In [30]:
def calc_loss_single_batch(input_batch, target_batch, model, device):
    # physically copies the data from our system RAM to 
    # the memory of our GPU (the "device").
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(
        logits.flatten(0, 1), target_batch.flatten()
    )
    return loss

In [41]:
def calc_avg_loss_multiple_batches(data_loader, model, device, num_batches=None):
    total_loss = 0
    batches_in_data_loader = len(data_loader)
    # if data_loader is empty, return error
    if batches_in_data_loader == 0:
        return float("nan")
    # if num_batches is unspecified, pick all batches
    if num_batches is None:
        batches_to_process = batches_in_data_loader
    else:
        batches_to_process = min(batches_in_data_loader, num_batches)

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < batches_to_process:
            loss = calc_loss_single_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break

    return total_loss / batches_to_process

### Picking the best available hardware (GPU, Mac Silicon or CPU)

* By default, tensors in PyTorch (like the batches from `DataLoader`) live on our CPU
* However, NN like the GPT-124M model perform better on a GPU
* **Overall, three things must be on the same device:**
    * The model weights.
    * The input_batch (the tokens).
    * The target_batch (the labels used to calculate the loss).

In [36]:
# detect what hardware is available
# basically, detect the best fastest available processor
# on our computer so that your GPT model doesn't run at 
# a snail's pace on the CPU if a better option exists.

# check for an NVIDIA graphics card with CUDA drivers installed
# GOLD STANDARD
if torch.cuda.is_available():
    device = torch.device("cuda")
# check for apple GPUs built into M1, M2 or M3 chips
elif torch.backends.mps.is_available():
    # Only use GPU on a mac if the PyTorch supports it
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
# fallback to CPU
else:
    device = torch.device("cpu")

print(f"Using {device} device.")

Using cpu device.


In [37]:
# Move the model to the device identified above

model.to(device)

GPTModel(
  (sem_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformer_blocks): Sequential(
    (0): TransformerBlock(
      (layerNorm1): LayerNorm()
      (mha): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (dropout): Dropout(p=0.1, inplace=False)
      (layerNorm2): LayerNorm()
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (1): TransformerBlock(
      (layerNorm1): LayerNorm()
      (mha): MultiHeadAtten

### Calculate loss on the training and validation batches, before training

In [42]:
torch.manual_seed(123) # For reproducibility due to the shuffling in the data loader

with torch.no_grad(): # Disable gradient tracking for efficiency because we are not training, yet
    train_loss = calc_avg_loss_multiple_batches(train_loader, model, device)
    val_loss = calc_avg_loss_multiple_batches(val_loader, model, device)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: 10.987583584255642
Validation loss: 10.98110580444336
